In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

# Define the state structure
class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

# Define node a
def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

# Define node b
def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}

# Initialize the StateGraph with the State schema
workflow = StateGraph(State)

# Add nodes to the graph
workflow.add_node("node_a", node_a)
workflow.add_node("node_b", node_b)

# Define edges and flow sequence
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

# Set up the memory checkpointer and compile the graph
checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# Pass the config into invoke here:
graph.invoke({"foo": "", "bar": []}, config)

{'foo': 'b', 'bar': ['a', 'b']}

In [4]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# Pass the config into invoke here:
graph.invoke({"foo": "", "bar": []}, config)
graph.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b', 'a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b34af-76f8-6b14-8006-fac83d202d73'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-09-18T10:23:18.543029+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b34af-76f3-6ce4-8005-335a3afe5acb'}}, tasks=(), interrupts=())